# 10. Historial laboral de personas

Proyecto de tesis (Maestria en Ciencia de Datos e IA, ESPOL): *Sistema de Generacion de Perfiles del Personal Docente y Administrativo en ESPOL para la asignacion inteligente de tareas*. Este notebook corresponde a la **Fase 1 y 2** de la metodologia: extraccion y depuracion de una de las fuentes institucionales que alimentan el catalogo de variables (historia laboral, capacitaciones, proyectos, experiencia y direcciones de tesis) usado para construir los perfiles multidimensionales del personal.

**Fuente:** `data/raw/historialaboralpersonas.csv`  
**Salida:** `data/processed/historial_laboral_personas.csv`, `data/processed/historial_laboral_periodos_continuos.csv`, `data/processed/historial_laboral_features.csv`

Historial de contratos del personal en ESPOL (cargo, tipo de contrato, fechas, unidad). En este extracto contiene los contratos de una persona a modo de muestra.

`IDPERSONA`, `IDCONTRATOLABORAL`, `NOMBRETABLA`, `IDUBICACIONFISICAEMP`, `TIPOEMPLEADO`, `NOMBRE_UNIDAD` (renombrada desde `NOMBRE` del CSV crudo: nombre de la unidad) e `IDESTRUCTURAORGANICA` se conservan siempre (no se eliminan aunque resulten constantes en la muestra) porque son claves o categorias necesarias para cruzar este historial con el resto de fuentes del proyecto. `NOMBRE_UNIDAD` se rellena con `DESCONOCIDA` cuando llega vacío (ver sección 3). `TIPOEMPLEADO` e `IDREGIMENLABORAL` se decodifican en columnas `_DESC` adicionales (ver seccion 6). La sección 10 genera un subdataset con los periodos continuos de vinculación por persona, y la sección 11 una tabla de features por persona (antigüedad, evolución de RMU, cargos, dedicación docente, movilidad, régimen) lista para clustering/perfilamiento.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [2]:
df = pc.leer_csv('historialaboralpersonas.csv', low_memory=False)
df.head()

Leido historialaboralpersonas.csv con encoding=utf-8-sig -> 73915 filas, 37 columnas


,NOMBRE,IDESTRUCTURAORGANICA,IDPERSONA,IDENTIFICACION,NOMBRES,APELLIDOS,IDCONTRATOLABORAL,CARGO,IDCARGO,CONTRATO,ESTADOCONTRATO,ESTADO,FECHAINICIOCONTRATO,FECHAFINCONTRATO,NOMBRETABLA,IDUNIDAD,MODULO,HORARIOFLEXIBLE,IDREGIMENLABORAL,TIPOEMPLEADO,FECHAVIGENCIA,FECHADESVINCULACION,HORAS,FECHAINGRESOESPOL,TIPODEDICACION,NROCONTRATO,ESPORCONCURSO,NIVELDOCENCIA,RMU,IDUBICACIONFISICAEMP,REFARCHIVO1,REFARCHIVO2,TIPO,FECHAREGISTRO,IDUSUARIO,IDUNIDADACADEMICAFIS,TIPOVINCULO
0,NaN,NaN,1750,0915244669,ROMEO ELIAS,CABRERA AREVALO,8054,PRESTACIÓN SERVICIOS PROFESIONALES,NaN,CONTRATO CIVIL,FF,FINALIZADO,2022-03-05,2022-03-13,TBL_CONTRATACION_OTROS,4.0,N,N,0,DD,2022-03-13,2022-03-13,4.0,NaN,Tiempo Parcial,FIEC-028-2022,NO,DOCENTE POSGRADO,160.0,NaN,"666585,625754",NaN,V,2022-02-25,79972.0,15004.0,SIN REL. DEPENDENCIA
1,NaN,NaN,1098,0909693400,DOUGLAS ANTONIO,PLAZA GUINGLA,5897,PRESTACIÓN SERVICIOS PROFESIONALES,NaN,CONTRATO CIVIL,FF,FINALIZADO,2020-04-03,2020-04-25,TBL_CONTRATACION_OTROS,4.0,N,N,0,DD,2020-04-25,2020-04-25,32.0,2003-10-20,Tiempo Parcial,FIEC-007-2020,NO,DOCENTE POSGRADO,1600.0,NaN,"474502,476922",NaN,V,2020-03-25,656079.0,15004.0,SIN REL. DEPENDENCIA
2,NaN,NaN,656058,122643548,LUIS ALBERTO,VILCAHUAMAN CAJACURI,6885,PRESTACIÓN SERVICIOS PROFESIONALES,NaN,CONTRATO CIVIL,FF,FINALIZADO,2021-03-05,2021-03-14,TBL_CONTRATACION_OTROS,4.0,N,N,0,DD,2021-03-14,2021-03-14,36.0,NaN,Tiempo Parcial,FIEC-013-2021,NO,DOCENTE POSGRADO,3600.0,NaN,"420576,423139",NaN,V,2021-03-02,79972.0,15004.0,SIN REL. DEPENDENCIA
3,NaN,NaN,102964,0962523270,JOHNNY WLADIMIR,RENGIFO SANTANA,6886,PRESTACIÓN SERVICIOS PROFESIONALES,NaN,CONTRATO CIVIL,FF,FINALIZADO,2021-03-06,2021-03-28,TBL_CONTRATACION_OTROS,4.0,N,N,0,DD,2021-03-28,2021-03-28,18.0,NaN,Tiempo Parcial,FIEC-014-2021,NO,DOCENTE POSGRADO,720.0,NaN,"418081,420645",NaN,V,2021-03-02,79972.0,15004.0,SIN REL. DEPENDENCIA
4,NaN,NaN,1043,0909187148,CARLOS TEODORO,MONSALVE ARTEAGA,6427,PRESTACIÓN SERVICIOS PROFESIONALES,NaN,CONTRATO CIVIL,FF,FINALIZADO,2020-12-04,2021-01-10,TBL_CONTRATACION_OTROS,4.0,N,N,0,DD,2021-01-10,2021-01-10,48.0,1990-01-11,Tiempo Parcial,007-MSIG-2021,NO,DOCENTE POSGRADO,3000.0,NaN,"462600,465043",NaN,V,2021-01-05,51775.0,15004.0,SIN REL. DEPENDENCIA


## 2. Exploración inicial

In [3]:
pc.resumen(df, 'historial_laboral_personas')

--- Resumen historial_laboral_personas ---
Dimensiones: 73915 filas x 37 columnas
Filas duplicadas: 3
Columnas con nulos (%):
FECHAVIGENCIA           77.1
NIVELDOCENCIA           57.6
REFARCHIVO2             49.1
FECHADESVINCULACION     46.8
TIPODEDICACION          44.4
IDCARGO                 29.8
NOMBRE                  25.4
IDESTRUCTURAORGANICA    25.4
IDUBICACIONFISICAEMP    25.3
FECHAINGRESOESPOL       23.8
IDUNIDADACADEMICAFIS    23.0
IDUNIDAD                19.0
IDUSUARIO                4.4
REFARCHIVO1              2.5
HORAS                    2.1
FECHAFINCONTRATO         1.9
CARGO                    0.7
FECHAREGISTRO            0.5
NROCONTRATO              0.0
CONTRATO                 0.0
dtype: float64


In [4]:
df.dtypes

NOMBRE                   object
IDESTRUCTURAORGANICA    float64
IDPERSONA                 int64
IDENTIFICACION           object
NOMBRES                  object
APELLIDOS                object
IDCONTRATOLABORAL         int64
CARGO                    object
IDCARGO                 float64
CONTRATO                 object
ESTADOCONTRATO           object
ESTADO                   object
FECHAINICIOCONTRATO      object
FECHAFINCONTRATO         object
NOMBRETABLA              object
IDUNIDAD                float64
MODULO                   object
HORARIOFLEXIBLE          object
IDREGIMENLABORAL          int64
TIPOEMPLEADO             object
FECHAVIGENCIA            object
FECHADESVINCULACION      object
HORAS                   float64
FECHAINGRESOESPOL        object
TIPODEDICACION           object
NROCONTRATO              object
ESPORCONCURSO            object
NIVELDOCENCIA            object
RMU                     float64
IDUBICACIONFISICAEMP    float64
REFARCHIVO1              object
REFARCHI

### Valores atípicos en RMU

Antes de limpiar, se revisan los casos más extremos de `RMU` con el método del rango intercuartílico (`pc.detectar_outliers_iqr`, factor 1.5, el criterio clásico de boxplot) para decidir si son errores de digitación, casos legítimos poco frecuentes (p.ej. autoridades, cargos directivos) u otra cosa. El mismo método sirve para revisar otras columnas numéricas si hiciera falta (`HORAS`, etc.).

In [5]:
print(df['RMU'].describe())

mask_outliers, limite_inferior, limite_superior = pc.detectar_outliers_iqr(df, 'RMU')
print(f"\nLimites IQR (factor 1.5): [{limite_inferior:.2f}, {limite_superior:.2f}]")
print(f"Casos atipicos: {mask_outliers.sum()} de {len(df)}")

columnas_revision = [c for c in ['IDPERSONA', 'IDCONTRATOLABORAL', 'CARGO', 'FECHAINICIOCONTRATO', 'RMU'] if c in df.columns]
df.loc[mask_outliers, columnas_revision].sort_values('RMU')

count    7.391500e+04
mean     5.854084e+04
std      7.617215e+05
min      0.000000e+00
25%      7.330000e+02
50%      1.412000e+03
75%      2.034000e+03
max      6.000000e+07
Name: RMU, dtype: float64

Limites IQR (factor 1.5): [-1218.50, 3985.50]
Casos atipicos: 9183 de 73915


,IDPERSONA,IDCONTRATOLABORAL,CARGO,FECHAINICIOCONTRATO,RMU
67519,851,7522,PROFESOR AGREGADO,2012-05-01,3989.00
70901,1060,11616,PROFESOR AGREGADO,2010-01-01,3995.00
14647,77575,51071,SUBDECANO(A),2026-04-20,3996.18
18777,3829,26578,SUBDECANO(A),2023-03-21,3996.18
59837,77575,48829,SUBDECANO(A),2026-01-27,3996.18
...,...,...,...,...,...
69904,973,10448,PROFESOR PREGRADO,1997-09-01,36000000.00
70327,6847,10963,PROFESOR,1995-05-01,40646400.00
63380,1001,2899,"INVESTIGADOR DIRECTOR DE TESIS, DIPLOMADO Y MA...",1999-11-24,50000000.00
70326,6847,10962,PROFESOR,1996-05-01,54000000.00


Estos casos son solo para revisión manual — no se eliminan ni se corrigen automáticamente, porque un RMU alto puede ser perfectamente legítimo (autoridades, cargos directivos, docentes titulares con muchos años) y no necesariamente un error. Si al revisar `IDCONTRATOLABORAL` contra la fuente institucional se confirma un error real (p.ej. un digito de más), corríjase puntualmente aquí antes de continuar con la limpieza, por ejemplo:

```python
correcciones_rmu = {}  # {IDCONTRATOLABORAL: valor_correcto}
if correcciones_rmu:
    df['RMU'] = df['IDCONTRATOLABORAL'].map(correcciones_rmu).fillna(df['RMU'])
```

## 3. Limpieza

`NOMBRE` (nombre de la unidad) se renombra a `NOMBRE_UNIDAD` para mayor claridad, y se rellena con `DESCONOCIDA` cuando llega vacío en el CSV crudo, en vez de eliminarse o dejarse nulo, para no perder la fila en agregaciones por unidad más adelante. `IDESTRUCTURAORGANICA` (identificador vigente de unidad, más confiable que `IDUNIDAD`/`IDUNIDADACADEMICAFIS`) también se protege de eliminarse por constante.

In [6]:
df = df.rename(columns={'NOMBRE': 'NOMBRE_UNIDAD'})
df = pc.limpiar_strings(df)
df = pc.rellenar_categoricas_nulas(df, ['NOMBRE_UNIDAD'], valor='DESCONOCIDA')
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(
    df,
    excluir=[
        'IDPERSONA', 'IDCONTRATOLABORAL', 'NOMBRETABLA', 'IDUBICACIONFISICAEMP',
        'TIPOEMPLEADO', 'NOMBRE_UNIDAD', 'IDESTRUCTURAORGANICA',
    ],
)

Columnas eliminadas por ser constantes: ['HORARIOFLEXIBLE']


## 5. Tipado de fechas e identificadores

In [7]:
df = pc.castear_fechas(df, ['FECHAINICIOCONTRATO', 'FECHAFINCONTRATO', 'FECHAVIGENCIA', 'FECHADESVINCULACION', 'FECHAINGRESOESPOL', 'FECHAREGISTRO'])
df = pc.castear_enteros(df, ['IDPERSONA', 'IDCONTRATOLABORAL', 'IDCARGO', 'IDUNIDAD', 'IDUBICACIONFISICAEMP', 'IDUSUARIO', 'IDUNIDADACADEMICAFIS', 'IDESTRUCTURAORGANICA'])

Nota: `FECHAFINCONTRATO` (y otras fechas de fin/desvinculacion) quedan como `NaT` cuando no existen en la fuente; eso indica que el contrato sigue vigente y no se elimina ni se invalida la fila.

## 6. Decodificación de catálogos

`TIPOEMPLEADO` (AA=Administrativo, DD=Docente) e `IDREGIMENLABORAL` (0=Contrato civil, 7=LOSEP, 8=LOES, 9=Código de Trabajo) se decodifican en columnas `_DESC` adicionales.

In [8]:
df = pc.decodificar_historial_laboral(df)
df[['TIPOEMPLEADO', 'TIPOEMPLEADO_DESC', 'IDREGIMENLABORAL', 'IDREGIMENLABORAL_DESC']].drop_duplicates()

,TIPOEMPLEADO,TIPOEMPLEADO_DESC,IDREGIMENLABORAL,IDREGIMENLABORAL_DESC
0,DD,DOCENTE,0,CONTRATO CIVIL
2770,AA,ADMINISTRATIVO,0,CONTRATO CIVIL
2823,NN,NaN,0,CONTRATO CIVIL
11134,AA,ADMINISTRATIVO,7,LOSEP - LEY ORGANICA DE SERVICIO PUBLICO
11137,DD,DOCENTE,8,LOES - LEY ORGANICA DE EDUCACION SUPERIOR
11202,AA,ADMINISTRATIVO,9,CT - CODIGO DE TRABAJO
60188,DD,DOCENTE,2,NaN
60224,AA,ADMINISTRATIVO,2,NaN
60229,DD,DOCENTE,3,NaN
60232,AA,ADMINISTRATIVO,4,NaN


## 7. Verificación final

In [9]:
pc.resumen(df, 'historial_laboral_personas (procesado)')
df.head()

--- Resumen historial_laboral_personas (procesado) ---
Dimensiones: 73915 filas x 38 columnas
Filas duplicadas: 3
Columnas con nulos (%):
FECHAVIGENCIA            77.1
NIVELDOCENCIA            57.6
REFARCHIVO2              49.1
FECHADESVINCULACION      46.8
TIPODEDICACION           44.4
IDCARGO                  29.8
IDESTRUCTURAORGANICA     25.4
IDUBICACIONFISICAEMP     25.3
FECHAINGRESOESPOL        23.8
IDUNIDADACADEMICAFIS     23.0
IDUNIDAD                 19.0
IDREGIMENLABORAL_DESC    12.6
IDUSUARIO                 4.4
REFARCHIVO1               2.5
HORAS                     2.1
FECHAFINCONTRATO          1.9
CARGO                     0.7
FECHAREGISTRO             0.5
TIPOEMPLEADO_DESC         0.1
NROCONTRATO               0.0
CONTRATO                  0.0
dtype: float64


,NOMBRE_UNIDAD,IDESTRUCTURAORGANICA,IDPERSONA,IDENTIFICACION,NOMBRES,APELLIDOS,IDCONTRATOLABORAL,CARGO,IDCARGO,CONTRATO,ESTADOCONTRATO,ESTADO,FECHAINICIOCONTRATO,FECHAFINCONTRATO,NOMBRETABLA,IDUNIDAD,MODULO,IDREGIMENLABORAL,TIPOEMPLEADO,FECHAVIGENCIA,FECHADESVINCULACION,HORAS,FECHAINGRESOESPOL,TIPODEDICACION,NROCONTRATO,ESPORCONCURSO,NIVELDOCENCIA,RMU,IDUBICACIONFISICAEMP,REFARCHIVO1,REFARCHIVO2,TIPO,FECHAREGISTRO,IDUSUARIO,IDUNIDADACADEMICAFIS,TIPOVINCULO,TIPOEMPLEADO_DESC,IDREGIMENLABORAL_DESC
0,DESCONOCIDA,<NA>,1750,0915244669,ROMEO ELIAS,CABRERA AREVALO,8054,PRESTACIÓN SERVICIOS PROFESIONALES,<NA>,CONTRATO CIVIL,FF,FINALIZADO,2022-03-05,2022-03-13,TBL_CONTRATACION_OTROS,4,N,0,DD,2022-03-13,2022-03-13,4.0,NaT,Tiempo Parcial,FIEC-028-2022,NO,DOCENTE POSGRADO,160.0,<NA>,"666585,625754",<NA>,V,2022-02-25,79972,15004,SIN REL. DEPENDENCIA,DOCENTE,CONTRATO CIVIL
1,DESCONOCIDA,<NA>,1098,0909693400,DOUGLAS ANTONIO,PLAZA GUINGLA,5897,PRESTACIÓN SERVICIOS PROFESIONALES,<NA>,CONTRATO CIVIL,FF,FINALIZADO,2020-04-03,2020-04-25,TBL_CONTRATACION_OTROS,4,N,0,DD,2020-04-25,2020-04-25,32.0,2003-10-20,Tiempo Parcial,FIEC-007-2020,NO,DOCENTE POSGRADO,1600.0,<NA>,"474502,476922",<NA>,V,2020-03-25,656079,15004,SIN REL. DEPENDENCIA,DOCENTE,CONTRATO CIVIL
2,DESCONOCIDA,<NA>,656058,122643548,LUIS ALBERTO,VILCAHUAMAN CAJACURI,6885,PRESTACIÓN SERVICIOS PROFESIONALES,<NA>,CONTRATO CIVIL,FF,FINALIZADO,2021-03-05,2021-03-14,TBL_CONTRATACION_OTROS,4,N,0,DD,2021-03-14,2021-03-14,36.0,NaT,Tiempo Parcial,FIEC-013-2021,NO,DOCENTE POSGRADO,3600.0,<NA>,"420576,423139",<NA>,V,2021-03-02,79972,15004,SIN REL. DEPENDENCIA,DOCENTE,CONTRATO CIVIL
3,DESCONOCIDA,<NA>,102964,0962523270,JOHNNY WLADIMIR,RENGIFO SANTANA,6886,PRESTACIÓN SERVICIOS PROFESIONALES,<NA>,CONTRATO CIVIL,FF,FINALIZADO,2021-03-06,2021-03-28,TBL_CONTRATACION_OTROS,4,N,0,DD,2021-03-28,2021-03-28,18.0,NaT,Tiempo Parcial,FIEC-014-2021,NO,DOCENTE POSGRADO,720.0,<NA>,"418081,420645",<NA>,V,2021-03-02,79972,15004,SIN REL. DEPENDENCIA,DOCENTE,CONTRATO CIVIL
4,DESCONOCIDA,<NA>,1043,0909187148,CARLOS TEODORO,MONSALVE ARTEAGA,6427,PRESTACIÓN SERVICIOS PROFESIONALES,<NA>,CONTRATO CIVIL,FF,FINALIZADO,2020-12-04,2021-01-10,TBL_CONTRATACION_OTROS,4,N,0,DD,2021-01-10,2021-01-10,48.0,1990-01-11,Tiempo Parcial,007-MSIG-2021,NO,DOCENTE POSGRADO,3000.0,<NA>,"462600,465043",<NA>,V,2021-01-05,51775,15004,SIN REL. DEPENDENCIA,DOCENTE,CONTRATO CIVIL


In [10]:
# Snapshot con todas las columnas (antes de la seccion 8) para el feature
# engineering de la seccion 11, que necesita HORAS/TIPODEDICACION/IDUNIDAD/etc.
df_historial = df.copy()

## 8. Columnas no usadas

Se eliminan columnas que no aportan al perfilamiento (identificadores internos de sistema, campos operativos/administrativos de bajo valor analitico o redundantes con otras ya conservadas):

`IDCARGO`, `IDUNIDAD`, `MODULO`, `HORARIOFLEXIBLE`, `FECHAVIGENCIA`, `HORAS`, `FECHAINGRESOESPOL`, `ESPORCONCURSO`, `REFARCHIVO1`, `REFARCHIVO2`, `FECHAREGISTRO`, `IDUSUARIO`, `TIPOVINCULO`, `IDUNIDADACADEMICAFIS`

In [11]:
columnas_no_usadas = [
    'IDCARGO', 'IDUNIDAD', 'MODULO', 'HORARIOFLEXIBLE', 'FECHAVIGENCIA',
    'HORAS', 'FECHAINGRESOESPOL', 'ESPORCONCURSO', 'REFARCHIVO1', 'REFARCHIVO2',
    'FECHAREGISTRO', 'IDUSUARIO', 'TIPOVINCULO', 'IDUNIDADACADEMICAFIS',
]
df = df.drop(columns=[c for c in columnas_no_usadas if c in df.columns])
df.head()

,NOMBRE_UNIDAD,IDESTRUCTURAORGANICA,IDPERSONA,IDENTIFICACION,NOMBRES,APELLIDOS,IDCONTRATOLABORAL,CARGO,CONTRATO,ESTADOCONTRATO,ESTADO,FECHAINICIOCONTRATO,FECHAFINCONTRATO,NOMBRETABLA,IDREGIMENLABORAL,TIPOEMPLEADO,FECHADESVINCULACION,TIPODEDICACION,NROCONTRATO,NIVELDOCENCIA,RMU,IDUBICACIONFISICAEMP,TIPO,TIPOEMPLEADO_DESC,IDREGIMENLABORAL_DESC
0,DESCONOCIDA,<NA>,1750,0915244669,ROMEO ELIAS,CABRERA AREVALO,8054,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2022-03-05,2022-03-13,TBL_CONTRATACION_OTROS,0,DD,2022-03-13,Tiempo Parcial,FIEC-028-2022,DOCENTE POSGRADO,160.0,<NA>,V,DOCENTE,CONTRATO CIVIL
1,DESCONOCIDA,<NA>,1098,0909693400,DOUGLAS ANTONIO,PLAZA GUINGLA,5897,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2020-04-03,2020-04-25,TBL_CONTRATACION_OTROS,0,DD,2020-04-25,Tiempo Parcial,FIEC-007-2020,DOCENTE POSGRADO,1600.0,<NA>,V,DOCENTE,CONTRATO CIVIL
2,DESCONOCIDA,<NA>,656058,122643548,LUIS ALBERTO,VILCAHUAMAN CAJACURI,6885,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2021-03-05,2021-03-14,TBL_CONTRATACION_OTROS,0,DD,2021-03-14,Tiempo Parcial,FIEC-013-2021,DOCENTE POSGRADO,3600.0,<NA>,V,DOCENTE,CONTRATO CIVIL
3,DESCONOCIDA,<NA>,102964,0962523270,JOHNNY WLADIMIR,RENGIFO SANTANA,6886,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2021-03-06,2021-03-28,TBL_CONTRATACION_OTROS,0,DD,2021-03-28,Tiempo Parcial,FIEC-014-2021,DOCENTE POSGRADO,720.0,<NA>,V,DOCENTE,CONTRATO CIVIL
4,DESCONOCIDA,<NA>,1043,0909187148,CARLOS TEODORO,MONSALVE ARTEAGA,6427,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2020-12-04,2021-01-10,TBL_CONTRATACION_OTROS,0,DD,2021-01-10,Tiempo Parcial,007-MSIG-2021,DOCENTE POSGRADO,3000.0,<NA>,V,DOCENTE,CONTRATO CIVIL


In [17]:
pc.resumen(df, 'historial_laboral_personas (procesado)')
df.head()

--- Resumen historial_laboral_personas (procesado) ---
Dimensiones: 73915 filas x 25 columnas
Filas duplicadas: 3
Columnas con nulos (%):
NIVELDOCENCIA            57.6
FECHADESVINCULACION      46.8
TIPODEDICACION           44.4
IDESTRUCTURAORGANICA     25.4
IDUBICACIONFISICAEMP     25.3
IDREGIMENLABORAL_DESC    12.6
FECHAFINCONTRATO          1.9
CARGO                     0.7
TIPOEMPLEADO_DESC         0.1
NROCONTRATO               0.0
CONTRATO                  0.0
dtype: float64


,NOMBRE_UNIDAD,IDESTRUCTURAORGANICA,IDPERSONA,IDENTIFICACION,NOMBRES,APELLIDOS,IDCONTRATOLABORAL,CARGO,CONTRATO,ESTADOCONTRATO,ESTADO,FECHAINICIOCONTRATO,FECHAFINCONTRATO,NOMBRETABLA,IDREGIMENLABORAL,TIPOEMPLEADO,FECHADESVINCULACION,TIPODEDICACION,NROCONTRATO,NIVELDOCENCIA,RMU,IDUBICACIONFISICAEMP,TIPO,TIPOEMPLEADO_DESC,IDREGIMENLABORAL_DESC
0,DESCONOCIDA,<NA>,1750,0915244669,ROMEO ELIAS,CABRERA AREVALO,8054,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2022-03-05,2022-03-13,TBL_CONTRATACION_OTROS,0,DD,2022-03-13,Tiempo Parcial,FIEC-028-2022,DOCENTE POSGRADO,160.0,<NA>,V,DOCENTE,CONTRATO CIVIL
1,DESCONOCIDA,<NA>,1098,0909693400,DOUGLAS ANTONIO,PLAZA GUINGLA,5897,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2020-04-03,2020-04-25,TBL_CONTRATACION_OTROS,0,DD,2020-04-25,Tiempo Parcial,FIEC-007-2020,DOCENTE POSGRADO,1600.0,<NA>,V,DOCENTE,CONTRATO CIVIL
2,DESCONOCIDA,<NA>,656058,122643548,LUIS ALBERTO,VILCAHUAMAN CAJACURI,6885,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2021-03-05,2021-03-14,TBL_CONTRATACION_OTROS,0,DD,2021-03-14,Tiempo Parcial,FIEC-013-2021,DOCENTE POSGRADO,3600.0,<NA>,V,DOCENTE,CONTRATO CIVIL
3,DESCONOCIDA,<NA>,102964,0962523270,JOHNNY WLADIMIR,RENGIFO SANTANA,6886,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2021-03-06,2021-03-28,TBL_CONTRATACION_OTROS,0,DD,2021-03-28,Tiempo Parcial,FIEC-014-2021,DOCENTE POSGRADO,720.0,<NA>,V,DOCENTE,CONTRATO CIVIL
4,DESCONOCIDA,<NA>,1043,0909187148,CARLOS TEODORO,MONSALVE ARTEAGA,6427,PRESTACIÓN SERVICIOS PROFESIONALES,CONTRATO CIVIL,FF,FINALIZADO,2020-12-04,2021-01-10,TBL_CONTRATACION_OTROS,0,DD,2021-01-10,Tiempo Parcial,007-MSIG-2021,DOCENTE POSGRADO,3000.0,<NA>,V,DOCENTE,CONTRATO CIVIL


## 9. Guardado en data/processed

In [12]:
pc.guardar_procesado(df, 'historial_laboral_personas.csv')

Guardado: C:\Users\carlo\Documents\MCD2025\Tesis MCD IA\proyectotesis\Proyecto_Tesis\data\processed\historial_laboral_personas.csv (73915 filas x 25 columnas)


WindowsPath('C:/Users/carlo/Documents/MCD2025/Tesis MCD IA/proyectotesis/Proyecto_Tesis/data/processed/historial_laboral_personas.csv')

## 10. Periodos continuos de contratación

Fusiona los contratos de cada `IDPERSONA` en periodos continuos de vinculación con ESPOL, replicando la lógica de "historia laboral continua" del sistema origen (`AuxiliarObtenerHistoriaContinua`), implementada en `pc.calcular_periodos_continuos`:

- La fecha fin efectiva de cada contrato es `FECHADESVINCULACION` si existe; si no, `FECHAFINCONTRATO`; si ninguna existe, el contrato está vigente (sin fecha fin).
- Los contratos de cada persona se procesan ordenados por `FECHAINICIOCONTRATO`. Dos contratos consecutivos se fusionan en el mismo periodo si se solapan (un contrato contenido dentro del periodo actual se ignora, sin retroceder el fin), son exactamente contiguos (fin + 1 día = inicio siguiente), o dejan una brecha corta tolerada: el contrato termina en los últimos 3 días de un mes y el siguiente inicia el día 1 del mes calendario siguiente (corte administrativo de fin de mes, igual que en el sistema origen).
- Un periodo vigente (sin fecha fin) se preserva como vigente al fusionar; una fecha fin real nunca reemplaza una vigencia ya detectada.
- Cualquier otra brecha cierra el periodo actual y abre uno nuevo.

Salida: `data/processed/historial_laboral_periodos_continuos.csv`, un registro por periodo continuo (no por contrato).

In [13]:
periodos_continuos = pc.calcular_periodos_continuos(df)
pc.guardar_procesado(periodos_continuos, 'historial_laboral_periodos_continuos.csv')
periodos_continuos.sort_values(['IDPERSONA', 'PERIODO_INICIO']).head(10)

Guardado: C:\Users\carlo\Documents\MCD2025\Tesis MCD IA\proyectotesis\Proyecto_Tesis\data\processed\historial_laboral_periodos_continuos.csv (12110 filas x 7 columnas)


,IDPERSONA,PERIODO_INICIO,PERIODO_FIN,PERIODO_VIGENTE,PERIODO_DURACION_DIAS,N_CONTRATOS,IDCONTRATOLABORAL
0,26,1998-05-25,1998-10-02,False,131,1,3139
1,26,1998-10-19,1999-10-02,False,349,4,"3159,3162,3163,3160"
2,26,1999-10-18,2000-03-03,False,138,1,3164
3,26,2000-05-15,2000-10-06,False,145,1,3167
4,26,2000-10-10,2001-09-28,False,354,3,"3170,3174,3186"
5,26,2002-05-20,2002-09-27,False,131,2,"3187,3188"
6,26,2002-10-21,2003-03-07,False,138,2,"3193,3190"
7,26,2003-05-19,2003-09-27,False,132,2,"3194,3195"
8,26,2003-10-20,2004-03-08,False,141,2,"3197,3199"
9,26,2004-05-17,2004-09-25,False,132,2,"3200,3201"


In [14]:
# Diagnostico: cuantos periodos continuos resulto teniendo cada persona
# (1 = historial sin brechas reales; >1 = hubo al menos una desvinculacion real entre contratos)
periodos_continuos.groupby('IDPERSONA').size().value_counts().sort_index().rename('n_personas').rename_axis('n_periodos_continuos')

n_periodos_continuos
1     1797
2      800
3      399
4      253
5      187
6      104
7       81
8       67
9       37
10      39
11      31
12      31
13      14
14      17
15      14
16       8
17      11
18       6
19       8
20       6
21       7
22       6
23       3
24       2
25       3
26       4
28       3
29       4
31       1
32       3
34       2
35       1
38       2
Name: n_personas, dtype: int64

## 11. Features de historial laboral para clustering/perfilamiento

A partir de `df_historial` (contratos, con todas las columnas de origen) y `periodos_continuos` (sección 10), se construye una tabla con **una fila por `IDPERSONA`** — la granularidad que necesita el modelo de perfilamiento — usando `pc.construir_features_historial_laboral`:

- **Conteo de contratos:** `N_REGISTROS_HISTORIAL` es la cantidad cruda de filas del CSV para la persona (puede incluir movimientos administrativos que no son un contrato nuevo). `N_CONTRATOS_TOTAL` es más representativo: cuenta cambios reales de `CARGO` o de `RMU` a lo largo de la línea de tiempo — dos filas consecutivas con el mismo cargo y la misma RMU cuentan como un solo contrato, en vez de contar filas. (Esto solo usa RMU como señal interna de "hubo un cambio", no expone el monto — ver nota de RMU más abajo.)
- **Antigüedad y continuidad:** fecha de primer ingreso, antigüedad efectiva (suma de días de los `PERIODO_DURACION_DIAS` de `periodos_continuos`, sin contar brechas reales) y antigüedad de calendario, número de periodos continuos y de reingresos, y si tiene vinculación vigente — **todo calculado a partir de `periodos_continuos`, no de las fechas crudas de cada contrato**, para no confundir vacaciones/licencias u otras brechas cortas ya fusionadas (sección 10) con una desvinculación real.
- **Tipo de empleado:** rol actual, si ha sido docente y administrativo a la vez, y años de experiencia aproximados por rol (`ANIOS_EXPERIENCIA_DOCENTE` / `ANIOS_EXPERIENCIA_ADMINISTRATIVO`).

  *Cómo leer estas columnas:* es la suma de días de todos los contratos de ese rol (fin efectivo — o hoy si sigue vigente — menos inicio), dividida entre 365.25 (promedia los años bisiestos) y redondeada a 2 decimales. Es un número decimal de años, **no** años+meses: p.ej. `5.01` son ~5 años y ~4 días (`0.01 × 365.25 ≈ 3.65` días), y `0.50` equivale a medio año (~6 meses). Puede sobreestimarse levemente si hay contratos simultáneos del mismo rol, ya que no se fusiona solapamiento dentro de un mismo rol (a diferencia de la antigüedad general, que sí usa `periodos_continuos`).
- **Dedicación docente:** la más reciente, la más frecuente y cuántas distintas tuvo (solo si `TIPODEDICACION` sobrevive la limpieza, p.ej. no en muestras solo administrativas).
- **Cargos:** cantidad de cargos distintos, cargo actual, cargo más frecuente, y si en algún año calendario tuvo más de un cargo distinto (`MULTIPLES_CARGOS_MISMO_ANIO`).
- **Movilidad:** cantidad de unidades distintas por `IDESTRUCTURAORGANICA` (no por `IDUNIDAD`/`IDUNIDADACADEMICAFIS`, identificadores de estructuras anteriores menos confiables); nombre de la unidad actual (`NOMBRE_UNIDAD`); cantidad de facultades distintas y si en algún momento pasó por rectorado/vicerrectorado, detectado por texto en `NOMBRE_UNIDAD` (`"FACULTAD"` / `"RECTORADO"`).
- **Régimen laboral:** régimen inicial y actual, y cantidad de regímenes distintos por los que pasó.
- **RMU: omitida por ahora (`incluir_rmu=False`).** El histórico mezcla montos en sucres y en dólares por la transición monetaria de Ecuador (año 2000), sin una conversión/normalización todavía implementada — comparar o promediar RMU sin resolver eso primero produciría magnitudes y "crecimientos" sin sentido. `pc.construir_features_historial_laboral` soporta `incluir_rmu=True` para cuando se resuelva la normalización de moneda; mientras tanto, las columnas `RMU_*` no se incluyen en el dataset exportado.
- **Estabilidad contractual:** `PROPORCION_CONTRATOS_FINALIZADOS` = proporción de `ESTADOCONTRATO`='FF' **solo entre las filas con `TIPO`='V'** (vinculación), si la columna `TIPO` existe. `TIPO`='M' suele ser un movimiento (vacaciones, licencias, etc.) y no una desvinculación real, pero esa clasificación aún no está depurada del todo — por ahora solo se filtra por 'V', sin más tratamiento.

Limitaciones conocidas:
- Los años de experiencia por rol pueden sobreestimarse levemente si existen contratos simultáneos del mismo `TIPOEMPLEADO` (ver arriba).
- La detección de facultad/rectorado por texto en `NOMBRE_UNIDAD` es una heurística simple (substring, sin distinguir mayúsculas); conviene revisarla contra los valores reales una vez que se disponga de datos de más personas.
- La clasificación `TIPO`='M' (movimientos) aún no distingue vacaciones, licencias u otros casos; queda pendiente para una futura depuración.
- RMU queda pendiente de una normalización sucre/dólar antes de reincorporarse como feature.

Salida: `data/processed/historial_laboral_features.csv`, un registro por persona.

In [15]:
features_historial = pc.construir_features_historial_laboral(df_historial, periodos_continuos, incluir_rmu=False)
pc.guardar_procesado(features_historial, 'historial_laboral_features.csv')
features_historial.head()

Guardado: C:\Users\carlo\Documents\MCD2025\Tesis MCD IA\proyectotesis\Proyecto_Tesis\data\processed\historial_laboral_features.csv (3951 filas x 31 columnas)


,IDPERSONA,N_REGISTROS_HISTORIAL,N_CONTRATOS_TOTAL,N_CARGOS_DISTINTOS,CARGO_ACTUAL,CARGO_MAS_FRECUENTE,MULTIPLES_CARGOS_MISMO_ANIO,N_UNIDADES_DISTINTAS,UNIDAD_ACTUAL_NOMBRE,N_FACULTADES_DISTINTAS,PASO_POR_RECTORADO,N_REGIMENES_DISTINTOS,REGIMEN_INICIAL_DESC,REGIMEN_ACTUAL_DESC,TIPOEMPLEADO_ACTUAL_DESC,ES_DOCENTE_ADMIN_MIXTO,ANIOS_EXPERIENCIA_DOCENTE,ANIOS_EXPERIENCIA_ADMINISTRATIVO,DEDICACION_DOCENTE_ACTUAL,DEDICACION_DOCENTE_MAS_FRECUENTE,N_DEDICACIONES_DOCENTE_DISTINTAS,PROPORCION_CONTRATOS_FINALIZADOS,FECHA_PRIMER_INGRESO,FECHA_ULTIMO_PERIODO_FIN,N_PERIODOS_CONTINUOS,ANTIGUEDAD_EFECTIVA_DIAS,VIGENTE_ACTUALMENTE,ANTIGUEDAD_EFECTIVA_ANIOS,ANTIGUEDAD_CALENDARIO_DIAS,ANTIGUEDAD_CALENDARIO_ANIOS,N_REINGRESOS
0,26,88,58,10,PROFESOR TITULAR AGREGADO 3 (TC),SUBDECANO(A),True,3,FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE ...,2,False,2,NaN,LOES - LEY ORGANICA DE EDUCACION SUPERIOR,DOCENTE,True,17.03,3.87,Tiempo Completo,Tiempo Completo,3,0.96,1998-05-25,2026-09-01,11,6206,True,16.99,10327,28.27,10
1,38,18,11,3,PROFESOR HONORARIO,PROFESOR HONORARIO,True,2,FACULTAD DE INGENIERÍA EN CIENCIAS DE LA TIERRA,1,False,1,CONTRATO CIVIL,CONTRATO CIVIL,DOCENTE,True,5.25,0.08,Tiempo Parcial,Tiempo Completo,3,1.00,2015-05-04,2023-02-17,12,2069,False,5.66,2847,7.79,11
2,39,6,6,2,SERVICIOS PROFESIONALES - EJECUCIÓN DE ACTIVID...,SERVICIOS PROFESIONALES - EJECUCIÓN DE ACTIVID...,True,1,GERENCIA DE INFRAESTRUCTURA FÍSICA,0,False,1,CONTRATO CIVIL,CONTRATO CIVIL,ADMINISTRATIVO,False,0.00,2.53,<NA>,<NA>,0,1.00,2016-12-09,2023-04-30,5,924,False,2.53,2334,6.39,4
3,40,11,7,2,PROFESOR HONORARIO,PROFESOR HONORARIO,True,1,FACULTAD DE INGENIERÍA EN CIENCIAS DE LA TIERRA,1,False,1,CONTRATO CIVIL,CONTRATO CIVIL,DOCENTE,True,2.65,2.35,Tiempo Parcial,Tiempo Parcial,2,1.00,2018-02-01,2023-12-31,7,1824,False,4.99,2160,5.91,6
4,41,42,16,1,CHOFER,CHOFER,False,1,DIRECCIÓN DE SERVICIOS GENERALES,0,False,1,NaN,CT - CODIGO DE TRABAJO,ADMINISTRATIVO,False,0.00,30.30,<NA>,<NA>,0,1.00,1993-05-10,2023-09-30,2,10980,False,30.06,11101,30.39,1


In [16]:
pc.resumen(features_historial, 'historial_laboral_features')

--- Resumen historial_laboral_features ---
Dimensiones: 3951 filas x 31 columnas
Filas duplicadas: 0
Columnas con nulos (%):
DEDICACION_DOCENTE_MAS_FRECUENTE    41.7
DEDICACION_DOCENTE_ACTUAL           41.7
REGIMEN_INICIAL_DESC                19.2
REGIMEN_ACTUAL_DESC                  0.0
dtype: float64
